In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys

sys.path.append("..")  # Go back to base directory

from modules.graph import *
from modules.viewer3d import *

In [ ]:
def dh_batch_transform_torch(theta, d, a, alpha):
    theta = torch.as_tensor(theta, dtype=torch.float32)
    d     = torch.as_tensor(d, dtype=torch.float32, device=theta.device)
    a     = torch.as_tensor(a, dtype=torch.float32, device=theta.device)
    alpha = torch.as_tensor(alpha, dtype=torch.float32, device=theta.device)

    ct, st = torch.cos(theta), torch.sin(theta)
    ca, sa = torch.cos(alpha), torch.sin(alpha)

    cca = ca.expand_as(ct)
    ssa = sa.expand_as(ct)
    dd = d.expand_as(ct)

    zeros = torch.zeros_like(theta)
    ones = torch.ones_like(theta)
    T = torch.stack([
        torch.stack([ct, -st * ca,  st * sa, a * ct], dim=-1),
        torch.stack([st,  ct * ca, -ct * sa, a * st], dim=-1),
        torch.stack([zeros, ssa, cca, dd], dim=-1),
        torch.stack([zeros, zeros, zeros, ones], dim=-1)
    ], dim=-2)
    
    return T

def batch_forward_kinematics_all_frames_torch(dh_params, batch_joint_values):
    batch_size, n_joints = batch_joint_values.shape
    T = torch.eye(4, dtype=batch_joint_values.dtype, device=batch_joint_values.device).unsqueeze(0).repeat(batch_size, 1, 1)
    frames = []
    for i, (theta, d, a, alpha) in enumerate(dh_params):
        current_theta = batch_joint_values[:, i]
        current_d = torch.as_tensor(d, dtype=batch_joint_values.dtype, device=batch_joint_values.device)
        A_i = dh_batch_transform_torch(current_theta, current_d, a, alpha)
        T = torch.bmm(T, A_i)  # batch matrix multiplication
        frames.append(T.clone())  # keep a copy

    return frames

In [ ]:
L1 = 0.4
L2 = 0.3
L3 = 0.2

dh_params = [
    ('q', 0.0, L1, 0.0),
    ('q', 0.0, L2, 0.0),
    ('q', 0.0, L3, 0.0)
]

joint_limits = [
    [-np.pi, np.pi],
    [0, 2 * np.pi/3],
    [0, 2 * np.pi/3],
]

In [ ]:
def encode_angles_batch(angles_batch):
    # Get the original shape
    N, M = angles_batch.shape

    # Calculate sine and cosine for the entire batch at once
    sines = torch.sin(angles_batch)   # Shape: (N, M)
    cosines = torch.cos(angles_batch) # Shape: (N, M)

    # Stack them along a new last dimension to create pairs
    # Shape becomes (N, M, 2)
    sincos_pairs = torch.stack((sines, cosines), dim=2)

    # Reshape to interleave the sin and cos components
    # The view (N, -1) flattens the last two dimensions (M, 2) into one of size 2*M
    encoded_angles = sincos_pairs.view(N, -1) # or .reshape(N, -1)

    return encoded_angles

def decode_sincos_batch(encoded_angles_batch):
    # Get the original shape
    N, L = encoded_angles_batch.shape
    M = L // 2 
    sincos_pairs = encoded_angles_batch.view(N, M, 2)

    # Extract sine (at index 0) and cosine (at index 1) from the last dimension
    # The '...' selects all preceding dimensions (in this case, N and M)
    sines = sincos_pairs[..., 0]   # Shape: (N, M)
    cosines = sincos_pairs[..., 1] # Shape: (N, M)

    # Use atan2 to correctly calculate the angle in all four quadrants for the entire batch
    angles = torch.atan2(sines, cosines)

    return angles

In [ ]:
def make_dataset(dataset_size):
    # Randomize based on joint limits
    output_data = []
    for low, high in joint_limits:
        random_joint_values = np.random.uniform(
            low=low,
            high=high,
            size=dataset_size,
        )

        output_data.append(random_joint_values)
    output_data = np.array(output_data).T

    output_data = torch.FloatTensor(output_data)
    encoded_output_data = encode_angles_batch(output_data)
    input_data = batch_forward_kinematics_all_frames_torch(dh_params, output_data)[-1][:,:3,3]
    
    return input_data, output_data, encoded_output_data

In [ ]:
class IKNN(nn.Module):
    def __init__(self, num_joints, in_features=3, hidden_layers=3, layer_size=400):
        super().__init__()
        
        self.num_joints = num_joints
        out_features = 2 * num_joints
        self.hidden_layers = hidden_layers
        self.layer_size = layer_size
        self.layers = nn.ModuleList()

        self.layers.append(nn.Linear(in_features, layer_size))
        for _ in range(hidden_layers - 1):
            self.layers.append(nn.Linear(layer_size, layer_size))
        self.layers.append(nn.Linear(layer_size, out_features))

    def forward_encoded(self, x):
        # Forward through the layers
        for layer in self.layers[:-1]:
            x = F.relu(layer(x))
        raw_output = self.layers[-1](x)
        
        # Process each (cos, sin) pair
        processed_outputs = []
        for i in range(self.num_joints):
            joint_pair = raw_output[:, i*2 : i*2+2]
            joint_pair_tanh = torch.tanh(joint_pair)
            joint_pair_normalized = F.normalize(joint_pair_tanh, p=2, dim=1)
            processed_outputs.append(joint_pair_normalized)
            
        final_output = torch.cat(processed_outputs, dim=1)
        
        return final_output

    def forward_decoded(self, x):
        return decode_sincos_batch(self.forward_encoded(x))


In [ ]:
model = IKNN(num_joints=len(dh_params))
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 1000
dataset_size = 100000
losses = []

for i in range(epochs):
    # Train based on position
    x_train, y_train_decoded, y_train_encoded = make_dataset(dataset_size)
    y_pred_decoded = model.forward_decoded(x_train)

    x_pred = batch_forward_kinematics_all_frames_torch(dh_params, y_pred_decoded)[-1][:,:3,3]

    loss = criterion(x_pred, x_train)

    # Train based on angles
    #y_pred_encoded = model.forward_encoded(x_train)
    #loss = criterion(y_pred_encoded, y_train_encoded)

    losses.append(loss.detach().numpy())

    # Only prints every 10 epochs
    if i % 10 == 0:
        print(f"Epoch: {i}, Loss: {loss}")

    optimizer.zero_grad() # Resets the gradient
    loss.backward() # Computes gradient of loss for each neuron
    optimizer.step() # Update weights and biases based on the gradient

torch.save(model.state_dict(), "IKNN.pth")

In [ ]:
model = IKNN(len(dh_params))
model.load_state_dict(torch.load("IKNN.pth"))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
%matplotlib inline

hm_resolution = 0.01
hm_size = 1.0
coord_range = np.arange(-hm_size, hm_size, hm_resolution)
ref_positions = []
for x in coord_range:
    for y in coord_range:
        ref_positions.append([x, y, 0.0])
ref_positions = torch.FloatTensor(ref_positions)

with torch.no_grad():
    inf_joint_angles = model.forward_decoded(ref_positions)
inf_positions = batch_forward_kinematics_all_frames_torch(dh_params, inf_joint_angles)[-1][:,:3,3]

error_hm = torch.reshape(torch.norm(inf_positions - ref_positions, dim=1), (coord_range.size, coord_range.size)) * 100 # In cm
max_error = 1 # In cm
error_hm = torch.clip(error_hm, 0, max_error) 

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.plot(range(epochs), losses, color="royalblue")
ax1.set_title("Training Loss Over Epochs")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Error")
ax1.grid(True, linestyle='--', alpha=0.6)

im = ax2.imshow(error_hm, cmap="YlOrRd", interpolation="nearest")
ax2.set_title("Heatmap of IK Error")
fig.colorbar(im, ax=ax2, label="Error (cm)")
reach_limit = Circle(
    (coord_range.size/2, coord_range.size/2), 
    coord_range.size * (L1 + L2 + L3) / (2 * hm_size), 
    color="white", 
    fill=False, 
    linewidth=1,
    linestyle="--"
)
ax2.add_patch(reach_limit)

plt.tight_layout()
plt.show()


In [ ]:
'''from coppeliasim_zmqremoteapi_client import RemoteAPIClient

# Init client
client = RemoteAPIClient()  # Client object
sim = client.getObject("sim")  # Simulation object'''

In [ ]:
'''joint_handles = [
    sim.getObject("/J0"),
    sim.getObject("/J0/L1/J1"),
    sim.getObject("/J0/L1/J1/L2/J2")
]
eef_handle = sim.getObject("/J0/L1/J1/L2/J2/L3")
goal_handle = sim.getObject("/Goal")

# Simulation begins here
sim.startSimulation()

while not sim.getSimulationStopping():
    goal_position = [sim.getObjectPosition(goal_handle)]

    with torch.no_grad():
        infered_angles = model.forward_decoded(torch.FloatTensor(goal_position)).flatten().tolist()

    for j, a in zip(joint_handles, infered_angles):
        sim.setJointPosition(j, a)

    eef_position = sim.getObjectPosition(eef_handle)

# Simulation ends here
sim.stopSimulation()'''